# Entrega 1 — Análise Comparativa de Representações
**Ouvidoria Inteligente: Triagem Semântica de Manifestações Cidadãs**  
**Aluno:** Ricardo Silva Flores

Este notebook compara Bag-of-Words (BoW), TF-IDF e embeddings densos nos pares de manifestações definidos no enunciado.

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

with open('manifestacoes.json', 'r', encoding='utf-8') as arquivo:
    manifestacoes = json.load(arquivo)

df = pd.DataFrame(manifestacoes)
textos = df['texto'].tolist()
df[['id', 'categoria_oficial', 'texto']].head()

,id,categoria_oficial,texto
0,M001,infraestrutura,A rua Sete de Setembro está sem manutenção há ...
1,M002,meio ambiente,Moradores estão despejando lixo doméstico às m...
2,M003,infraestrutura,"Existe um buraco enorme na Av. Brasil, próximo..."
3,M004,educação,A Escola Municipal José de Alencar está sem pr...
4,M005,saúde,Sou moradora do bairro Vila Nova e gostaria de...


## Representações esparsas
BoW registra contagens de termos. TF-IDF reduz o peso de palavras muito frequentes e aumenta o peso de termos mais discriminativos.

In [2]:
bow = CountVectorizer(lowercase=True, strip_accents='unicode')
X_bow = bow.fit_transform(textos)

tfidf = TfidfVectorizer(lowercase=True, strip_accents='unicode', ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(textos)

print('Matriz BoW:', X_bow.shape)
print('Matriz TF-IDF:', X_tfidf.shape)

Matriz BoW: (40, 657)
Matriz TF-IDF: (40, 1835)


## Embeddings densos
O modelo multilíngue representa frases semanticamente parecidas em posições próximas no espaço vetorial. Se o download do modelo falhar, o notebook informa o erro; conecte-se à internet e execute novamente.

In [3]:
from sentence_transformers import SentenceTransformer

modelo_nome = 'paraphrase-multilingual-MiniLM-L12-v2'
modelo = SentenceTransformer(modelo_nome)
X_embeddings = modelo.encode(textos, show_progress_bar=True)
print('Matriz de embeddings:', X_embeddings.shape)

C:\Users\PICHAU\Documents\facul\8periodo\ricardo\OuvidoriaInteligente\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 2/2 [00:00<00:00,  4.51it/s]

Matriz de embeddings: (40, 384)


In [4]:
pares = [('M003', 'M017'), ('M008', 'M022'), ('M008', 'M031')]
indice_por_id = {codigo: i for i, codigo in enumerate(df['id'])}

resultados = []
for id_a, id_b in pares:
    i, j = indice_por_id[id_a], indice_por_id[id_b]
    resultados.append({
        'Par': f'{id_a} × {id_b}',
        'BoW': round(float(cosine_similarity(X_bow[i], X_bow[j])[0, 0]), 4),
        'TF-IDF': round(float(cosine_similarity(X_tfidf[i], X_tfidf[j])[0, 0]), 4),
        'Embeddings': round(float(cosine_similarity([X_embeddings[i]], [X_embeddings[j]])[0, 0]), 4),
    })

tabela_resultados = pd.DataFrame(resultados)
tabela_resultados

,Par,BoW,TF-IDF,Embeddings
0,M003 × M017,0.1886,0.0520,0.5373
1,M008 × M022,0.2619,0.0710,0.7005
2,M008 × M031,0.2294,0.0646,0.2827


In [5]:
for id_a, id_b in pares:
    print(f'\n{id_a}:', df.loc[df.id == id_a, 'texto'].iloc[0])
    print(f'{id_b}:', df.loc[df.id == id_b, 'texto'].iloc[0])


M003: Existe um buraco enorme na Av. Brasil, próximo ao número 450, que já causou danos a pneus de vários veículos que passam pelo local.
M017: O asfalto está todo esburacado na avenida principal do centro, dificultando o tráfego e causando vários acidentes de moto no local.

M008: O posto de saúde do bairro Alto do Mateus está sem médico há semanas, deixando a população sem atendimento clínico básico na região.
M022: Está faltando atendimento no PSF do meu bairro, os pacientes chegam cedo e não conseguem ser vistos por nenhum médico durante o dia.

M008: O posto de saúde do bairro Alto do Mateus está sem médico há semanas, deixando a população sem atendimento clínico básico na região.
M031: A lâmpada do poste em frente à praça central está queimada há semanas e ninguém da prefeitura veio fazer a substituição até agora.


## Conclusão
BoW e TF-IDF dependem da sobreposição de vocabulário. Por isso, eles podem subestimar textos que descrevem o mesmo problema com palavras diferentes, como buraco, asfalto danificado e via esburacada. Os embeddings tendem a capturar melhor relações de significado e devem atribuir valores maiores aos pares M003 × M017 e M008 × M022. Já M008 × M031 deve permanecer com baixa similaridade, pois trata de falta de atendimento médico versus iluminação pública.

A análise deve usar os valores gerados nesta execução. Eles podem variar levemente conforme a versão do modelo e do ambiente.